# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Entrenamiento de un agente de **Deep RL** para conducir en el simulador 3D
**Duckietown** usando solo píxeles de la cámara. La nota depende del rendimiento en un
**mapa oculto** con obstáculos (`Duckietown-loop_obstacles-v0`).

### 📋 Fases del proyecto
* **Fase 1 — Calentamiento:** Q-Learning tabular *desde cero* en `FrozenLake-v1` (8x8, resbaladiza).
* **Fase 2 — Baselines:** **DQN** (acción discreta) y **PPO** (acción continua) con Stable-Baselines3.
* **Fase 3 — Algoritmo avanzado:** **SAC** (Soft Actor-Critic, continuo).

### ⚠️ Contrato de evaluación (resumen)
1. Observación **`(1, 64, 64)`** por frame, apilada en **4 frames** → `(4, 64, 64)`.
2. Las clases (`CustomCNN`, wrappers) deben estar definidas e importables al cargar.
3. El profesor carga `best_agent.zip` en un entorno limpio (`requirements.txt`) y
   evalúa en el mapa oculto **`Duckietown-loop_obstacles-v0`** (entrenar ahí = descalificación).

### 🗂️ Organización del código (repositorio)
| Archivo | Contenido |
|---|---|
| `q_learning_sandbox.py` | Fase 1 (Q-Learning tabular) |
| `src/config.py` | Contrato: shapes, mapas, acciones discretas |
| `src/duckie_factory.py` | Entorno base real/mock (capa de abstracción) |
| `src/wrappers.py` | `DuckieWrapper` + `DiscreteActionWrapper` |
| `src/cnn.py` | `CustomCNN` (extractor de características para SB3) |
| `src/envs.py` | `make_env` + `build_vec_env` (FrameStack) |
| `train.py` | Entrenamiento DQN/PPO/SAC |
| `eval.py` | Evaluación de un modelo guardado |
| `scripts/run_training_plan.py` | Lanzador de experimentos |

> Detalle completo del entorno en **`COLAB_SETUP.md`**; experimentos y resultados en
> **`EXPERIMENTS.md`**.

> **Entrega autosuficiente:** este ZIP ya contiene todo (código, `requirements.txt`,
> `best_agent.zip` en la raíz, `Report.pdf`). Para evaluar: subir el ZIP a Colab (o
> descomprimirlo como `/content/MAML`), abrir este notebook y ejecutar las celdas en
> orden. La **primera celda descomprime el ZIP automáticamente** si solo está subido sin
> descomprimir. **No** hay que clonar GitHub ni subir el modelo aparte; el notebook usa
> los archivos locales y copia `best_agent.zip` a `models/` automáticamente.

## Dependencias y entorno (Colab limpio)

Este notebook se ejecuta **desde la carpeta del proyecto descomprimida del ZIP de
entrega** (idealmente `/content/MAML`). La primera celda **localiza los archivos
locales** (no clona GitHub por defecto; el clon queda como *fallback* de desarrollo).

El kernel de Colab es **Python 3.12**, pero el stack (numpy 1.23.5, gym 0.25.2,
Duckietown…) requiere **Python 3.11**. Por eso montamos un **venv 3.11** y ejecutamos
todo con `{PY}` (subprocess), no con el `pip`/`python` del kernel.

**Ejecutar las celdas de esta sección en orden.**

In [ ]:
# Usar los archivos LOCALES del ZIP de entrega (NO se clona GitHub por defecto).
# Si solo está MAML_entrega_final.zip subido (sin descomprimir), se descomprime solo.
import os
import zipfile

def _has_project(d):
    return (os.path.isfile(os.path.join(d, "requirements.txt"))
            and os.path.isfile(os.path.join(d, "train.py"))
            and os.path.isfile(os.path.join(d, "eval.py"))
            and os.path.isdir(os.path.join(d, "src")))

def _find_project():
    return next((d for d in ["/content/MAML", "/content", os.getcwd()]
                 if _has_project(d)), None)

PROJECT = _find_project()

# Si no se encuentra el proyecto, intentar descomprimir el ZIP de entrega.
if PROJECT is None:
    zip_path = next((z for z in ["/content/MAML_entrega_final.zip",
                                 os.path.join(os.getcwd(), "MAML_entrega_final.zip")]
                     if os.path.isfile(z)), None)
    if zip_path is not None:
        # Solo creamos/usamos /content/MAML si no es ya un proyecto válido.
        if not _has_project("/content/MAML"):
            os.makedirs("/content/MAML", exist_ok=True)
            print(f"Descomprimiendo {zip_path} en /content/MAML ...")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall("/content/MAML")
        PROJECT = _find_project()

if PROJECT is None:
    print("[ERROR] No se encontraron los archivos del proyecto "
          "(requirements.txt, train.py, eval.py, src/) ni MAML_entrega_final.zip.")
    print("Sube y/o descomprime el ZIP de entrega como /content/MAML y reejecuta.")
    print("Fallback de desarrollo (solo si NO tienes el ZIP): descomenta el git clone:")
    # !git clone https://github.com/AdolfoPM02/MAML.git /content/MAML
    raise FileNotFoundError("Proyecto no encontrado: descomprime el ZIP como /content/MAML.")

print("Proyecto encontrado en:", PROJECT)
%cd $PROJECT
!ls

In [ ]:
# a) Python 3.11 + dependencias de sistema (Duckietown headless)
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev xvfb python3-opengl ffmpeg freeglut3-dev

In [ ]:
# b) Crear el venv 3.11 y definir PY (intérprete del venv usado en TODO el notebook)
!python3.11 -m venv /content/venv-maml
PY = "/content/venv-maml/bin/python"
!{PY} -m pip install -U pip wheel setuptools
!{PY} --version

In [ ]:
# c) Instalar el stack EXACTO con el Python 3.11 del venv (NO el kernel 3.12)
%cd $PROJECT
!{PY} -m pip install -r requirements.txt
# gym-duckietown se instala APARTE y SIN deps (sus dependencias ya van en requirements.txt)
!{PY} -m pip install --no-deps "duckietown-gym-daffy @ git+https://github.com/duckietown/gym-duckietown.git@c76a7fec7f739db4f624f40ca83ce99383665558"
# Re-fijar numpy al final por si alguna dependencia lo cambió
!{PY} -m pip install --force-reinstall --no-deps numpy==1.23.5

In [ ]:
# d) Verificación de imports y versiones
!MPLBACKEND=Agg {PY} -c "import numpy, torch, gym, gymnasium, stable_baselines3; import gym_duckietown; print('numpy', numpy.__version__); print('torch', torch.__version__); print('gym', gym.__version__); print('gymnasium', gymnasium.__version__); print('stable_baselines3', stable_baselines3.__version__); print('import gym_duckietown OK')"

## Fase 1 — Q-Learning tabular (`FrozenLake-v1`)
Implementación **desde cero** (sin librerías de Deep RL) de Q-Learning tabular sobre
`FrozenLake-v1` 8x8 resbaladiza: Q-table en numpy, actualización por la **Ecuación de
Bellman**

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\,]$$

con política **ε-greedy** y decaimiento, y gráfica de media móvil de recompensa.
Implementación completa en **`q_learning_sandbox.py`** (celda **opcional**, ~30 s):

In [ ]:
!MPLBACKEND=Agg {PY} q_learning_sandbox.py

## Fase 2 — Baselines DQN y PPO (Duckietown)

**Pipeline de visión** (`src/wrappers.py` → `DuckieWrapper`):

`RGB (480×640×3)` → **recorte** del 50% superior (cielo) → **escala de grises** →
**resize 64×64** → `(1, 64, 64)` → **FrameStack(4)** (`src/envs.py`) → `(4, 64, 64)` →
**`CustomCNN`** (`src/cnn.py`).

**Baselines:**
* **DQN** — acción **discreta**: `DiscreteActionWrapper` mapea cada índice a un comando
  `[velocidad, giro]` (`src/config.DISCRETE_ACTIONS`).
* **PPO** — acción **continua**: `Box([-1,-1], [1,1])`.

Ambos usan `CustomCNN` vía `policy_kwargs`. El entrenamiento real se lanza con
`scripts/run_training_plan.py` (ver `EXPERIMENTS.md`); aquí solo los comandos en
**dry-run** (no entrena):

In [ ]:
!{PY} scripts/run_training_plan.py --stage ppo20k --dry-run --eval-after
!{PY} scripts/run_training_plan.py --stage dqn20k --dry-run --eval-after

## Fase 3 — Algoritmo avanzado (SAC)
**SAC** (Soft Actor-Critic) como algoritmo avanzado de acción continua (off-policy,
exploración por entropía). Comparte `CustomCNN` y wrappers con PPO. Dry-run (no entrena):

In [ ]:
!{PY} scripts/run_training_plan.py --stage sac20k --dry-run --eval-after

> **Nota:** aunque SAC se implementó como alternativa avanzada, el **modelo final
> seleccionado fue PPO (20k)** por sus resultados (ver tabla siguiente).

## Resultados finales
Entrenamiento real en Colab (GPU) con el stack validado y `--init-order model-first`.
Evaluación con `eval.py` (3 episodios por mapa). Celda = `recompensa media ± std / longitud media`.

| modelo | ts | loop_empty | small_loop | loop_obstacles | decisión |
|---|---|---|---|---|---|
| DQN 20k | 20k | −1013.4 ± 27.0 / 112 | −1001.9 ± 46.8 / 86 | −1065.8 ± 20.1 / 61 | descartado |
| DQN 50k | 50k | −1047.6 ± 76.9 / 82 | −1016.6 ± 65.7 / 74 | −1032.0 ± 52.4 / 37 | descartado |
| **PPO 20k** | 20k | **961.0 ± 605.4 / 1500** | **317.5 ± 703.3 / 1500** | **1118.2 ± 488.2 / 1500** | **GANADOR** |
| PPO 50k | 50k | −813.7 ± 210.5 / 1500 | −1238.9 ± 393.6 / 1500 | −1105.4 ± 760.6 / 1500 | descartado |
| SAC 20k | 20k | 990.6 ± 139.3 / 1500 | −1253.6 ± 120.3 / 296 | 915.2 ± 571.9 / 1500 | evaluado, no supera a PPO 20k |
| SAC 50k | 50k | — | — | — | **no completado** (ver nota) |

**Ganador: `ppo_loop_empty_20k_gpu`** (entregado como `best_agent.zip`): mejor equilibrio
global y la **mejor recompensa en el mapa oculto `loop_obstacles`** (1118.2), por encima de
SAC 20k (915.2) y de todo DQN.

- **DQN (20k/50k):** baseline discreto **descartado** — recompensas negativas y episodios
  muy cortos (longitudes 37–112 ≪ 1500); entrenar más no mejora.
- **PPO:** 20k gana; **PPO 50k descartado** (más entrenamiento degradó la política).
- **SAC 20k** (Fase 3): **implementado, entrenado y evaluado**; bueno en `loop_empty` y
  `loop_obstacles`, pero **mala generalización en `small_loop`** y **no supera a PPO 20k**.

> **Nota — SAC 50k (no completado):** la ejecución se **interrumpió en Colab hacia 30k
> timesteps tras más de 6 h** y **no generó `sac_advanced_50k.zip`**. Por reproducibilidad
> **no se reporta como métrica cuantitativa**, solo como intento no completado.

`Duckietown-loop_obstacles-v0` se usó **solo para evaluación** (`--allow-eval-hidden`),
**nunca para entrenamiento** (bloqueado por `ValueError` en el código).

## Modelo final
El entregable del contrato es **`best_agent.zip`**.

**`best_agent.zip` = copia de `ppo_loop_empty_20k_gpu.zip`** (~4.6 MB), generado en
Colab con `cp models/ppo_loop_empty_20k_gpu.zip models/best_agent.zip`. *(Durante el
desarrollo se usó el nombre `best_duckie_agent.zip`; el entregable definitivo es
`best_agent.zip`.)*

La siguiente celda **prepara el modelo automáticamente**: si `best_agent.zip` está en la
raíz del proyecto (como en el ZIP de entrega), lo copia a `models/best_agent.zip`; si ya
está en `models/`, no hace nada; si no aparece en ninguna ruta, lanza un error claro.

In [ ]:
# Preparar best_agent.zip automáticamente (viene en la raíz del ZIP de entrega)
%cd $PROJECT

import os
import shutil

os.makedirs("models", exist_ok=True)

if os.path.exists("best_agent.zip"):
    shutil.copy("best_agent.zip", "models/best_agent.zip")
    print("OK: best_agent.zip copiado a models/best_agent.zip")
elif os.path.exists("models/best_agent.zip"):
    print("OK: models/best_agent.zip ya existe")
else:
    raise FileNotFoundError(
        "No se encontró best_agent.zip. Debe estar en la raíz del proyecto "
        "o en models/best_agent.zip."
    )

!ls -lh models/best_agent.zip

In [ ]:
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} eval.py --algo ppo --model models/best_agent --map Duckietown-loop_empty-v0 --episodes 1 --device cpu --init-order model-first

## 📦 Entrega
El ZIP final debe contener:

* **`Challenge_RL.ipynb`** — este notebook.
* **`requirements.txt`** — dependencias con versiones fijadas (`==`).
* **`q_learning_sandbox.py`** — Fase 1.
* **`train.py`** — pipeline de entrenamiento (DQN/PPO/SAC).
* **`best_agent.zip`** — modelo final (copia de `ppo_loop_empty_20k_gpu.zip`).
* **`Report.pdf`** — memoria técnica (comparativa de los 3 algoritmos y justificación de la Fase 3).

Opcionalmente, el resto del código: `eval.py`, `src/`, `scripts/`, `COLAB_SETUP.md`,
`EXPERIMENTS.md`.

> **En la entrega final, `best_agent.zip` se incluye en la raíz del ZIP. El notebook lo
> copia automáticamente a `models/best_agent.zip`** (celda de la sección *Modelo final*),
> por lo que no hay que subirlo a mano.

**Dry-run del contrato recomendado:** Colab limpio → descomprimir el ZIP como
`/content/MAML` → ejecutar la sección *Dependencias* → ejecutar la sección *Modelo final*
(copia automática + evaluación).